# Day 28 — Quantization, Visualized

Store weights as int8 instead of fp32: ~4x smaller and faster, for a tiny accuracy hit.

In [ ]:
import io, torch, torch.nn as nn
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 256),
                      nn.ReLU(), nn.Linear(256, 10)).eval()
qmodel = torch.ao.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)

def size_kb(m):
    buf = io.BytesIO(); torch.save(m.state_dict(), buf); return buf.getbuffer().nbytes / 1024
print(f'fp32: {size_kb(model):.1f} KB   int8: {size_kb(qmodel):.1f} KB')

In [ ]:
import matplotlib.pyplot as plt
x = torch.randn(8, 64)
with torch.no_grad(): err = (qmodel(x) - model(x)).abs().mean().item()
plt.figure(figsize=(5, 3.5))
plt.bar(['fp32', 'int8'], [size_kb(model), size_kb(qmodel)], color=['#888', '#55A868'])
plt.ylabel('KB'); plt.title(f'~4x smaller  (mean abs output error {err:.3f})')
plt.show()

## Takeaways

- Dynamic quantization -> int8 Linear weights: smaller + faster, small accuracy cost.
- Verify the output error stays tiny relative to the signal.
- This is a standard last step before shipping a model to constrained hardware.

**Now build** quantize / model_size_bytes in `homework.py`, then `pytest -q`.
You've now built the whole train -> export -> lower -> quantize deployment pipeline.